# Big Data Analytics — Progetto: Apache Spark RDD

## Obiettivo

Questo notebook esplora due funzionalità fondamentali di Apache Spark attraverso
il dataset pubblico **Divvy Bikes** (Chicago bike-sharing system).

### Parte 1  RDD e In-Memory Computing
Implementiamo l'algoritmo **K-Means clustering** manualmente con RDD puri,
applicato alle coordinate geografiche delle stazioni Divvy.

L'algoritmo viene eseguito in due modalità:
- **Senza `persist()`**: ad ogni iterazione Spark ricalcola l'RDD risalendo
  il lineage dal file sorgente, comportamento analogo al classico MapReduce,
  dove i dati intermedi vengono riletti da disco ad ogni passo.
- **Con `persist()`**: l'RDD viene materializzato in RAM alla prima computazione
  e riutilizzato nelle iterazioni successive, eliminando l'I/O ripetuto.

Il confronto mostra quantitativamente il vantaggio dell'in-memory computing
per algoritmi iterativi, che rappresentano il caso d'uso in cui Spark supera
più nettamente il paradigma MapReduce tradizionale.

### Parte 2 — Structured Streaming
Estendiamo l'elaborazione al caso **online**: simuliamo un flusso continuo di
nuovi trip Divvy e usiamo Spark Structured Streaming per aggiornare in
tempo reale le aggregazioni per stazione (trip attivi, stazioni più usate).

Questo scenario evidenzia i limiti dei DBMS relazionali classici per dati
in movimento e motiva l'uso di un framework come Spark anche per workload
di tipo streaming.

In [ ]:
%pip install pyspark==3.5.3


Note: you may need to restart the kernel to use updated packages.


Import

In [2]:
from pyspark import SparkContext, SparkConf
import os, time, zipfile
import numpy as np
import matplotlib.pyplot as plt

Download del dataset

In [3]:
URL  = "https://divvy-tripdata.s3.amazonaws.com/202409-divvy-tripdata.zip"
CSV_PATH = "202409-divvy-tripdata.csv"

Setup Spark

In [4]:
conf = SparkConf() \
    .setAppName("DivvyKMeans") \
    .setMaster("local[*]") \
    .set("spark.driver.memory", "2g")

sc = SparkContext(conf=conf)
sc.setLogLevel("ERROR")
print("Spark avviato:", sc.version)
print("UI disponibile su: http://localhost:4040")

Spark avviato: 4.1.2
UI disponibile su: http://localhost:4040


Lettura del csv
- parse_coords legge le righe e prende solo il campo "latitude, longitude"

In particolare start_lat e start_lng sono le coordinate della stazione di partenza del viaggio, il punto dove l'utente ha preso la bici.<br>
Per il K-Means l'idea è raggruppare i punti di partenza in cluster geografici per capire quali zone di Chicago hanno più domanda. <br>
Ogni punto nell'RDD rappresenta un viaggio, quindi le zone con più punti concentrati sono quelle con più utilizzo.

In [ ]:
raw_rdd = sc.textFile(CSV_PATH)
header  = raw_rdd.first()

def parse_coords(line):
    fields = line.split(",")
    try:
        lat = float(fields[8])
        lng = float(fields[9])
        if 41.6 <= lat <= 42.1 and -87.9 <= lng <= -87.5:  #  filtro rumore: bounding box Chicago
            return (lat, lng)
    except:
        pass
    return None

coords_rdd = raw_rdd \
    .filter(lambda line: line != header) \
    .map(parse_coords) \
    .filter(lambda x: x is not None)

n_points = coords_rdd.count()
print(f"Punti validi: {n_points:,}")

Punti validi: 821,275


Funzioni per implementazione del K-means

- dist_sq(a, b)
Calcola la distanza euclidea
- assign_cluster(point, centroids)
Dato un punto e la lista dei centroidi, restituisce l'indice del centroide più vicino. 
- kmeans computa iterativamente i centroidi fino a convergenza, il flag "use_cache" sarà usato per simulare RDD+in_memory (come fa SPARK) e confrontandolo con la versione precedente map/reduce che non supporta in memory.

In [ ]:
def dist_sq(a, b):
    return (a[0]-b[0])**2 + (a[1]-b[1])**2

def assign_cluster(point, centroids):
    return min(range(len(centroids)), key=lambda i: dist_sq(point, centroids[i]))

''' 
MAP: ogni punto viene trasformato in (cluster_id, (lat, lng, 1)) dove 1 è il contatore per calcolare la media dopo.
REDUCE: i punti dello stesso cluster vengono aggregati sommando lat, lng e contatore: (sum_lat, sum_lng, count).

In sostanza bisogna computare iterativamente una MEDIA per il calcolo del centroide, quindi per aggregare è necessario avere somma e 
contatore. Questo perchè la media NON è associativa
'''
def kmeans(rdd, k=10, max_iter=20, use_cache=False, label=""):
    centroids = rdd.takeSample(False, k, seed=42)       #primi k centroidi scelti casualmente
    
    if use_cache:
        rdd.persist()
        _ = rdd.count()  # forza materializzazione prima di misurare
    
    iter_times = []

    for i in range(max_iter):
        t0 = time.time()

        bc = sc.broadcast(centroids) #invia la lista dei centroidi a tutti i worker

        # MAP: ogni punto → (cluster_id, (lat, lng, 1))
        # REDUCE: somma per cluster → nuovo centroide 
        new_centroids_rdd = (
            rdd
            .map(lambda p: (assign_cluster(p, bc.value), (p[0], p[1], 1)))      #ad ogni punto assegni il cluster più vicino e trasformi in (cluster_id, (lat, lng, 1))
            .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1], a[2]+b[2]))   #reduceByKey per sommare lat, lng e contatore per ogni cluster_id
            .map(lambda kv: (kv[0], (kv[1][0]/kv[1][2], kv[1][1]/kv[1][2]))) #calcola il nuovo centroide come media: (sum_lat/count, sum_lng/count
        )

        new_map   = new_centroids_rdd.collectAsMap()
        new_cents = [new_map.get(j, centroids[j]) for j in range(k)]

        elapsed = time.time() - t0
        iter_times.append(elapsed)

        shift = max(dist_sq(centroids[j], new_cents[j]) for j in range(k))
        centroids = new_cents
        bc.unpersist()

        print(f"  [{label}] iter {i+1:2d} | {elapsed:.3f}s | shift: {shift:.2e}")

        if shift < 1e-9:        #convergenza se il delta tra i centroidi di questa it e la precedente è molto piccolo
            print(f"  Convergenza raggiunta all'iterazione {i+1}")
            break

    if use_cache:
        rdd.unpersist()

    return centroids, iter_times

Esecuzione del K-means :
- Map/reduce "nativo" senza in-memory  (nativo nel senso prima dell'arrivo di Spark e della tecnologia in-memory)
- Spark

In [ ]:
print("=== SENZA persist() — equivalente MapReduce ===")
cents_no, times_no = kmeans(coords_rdd, k=15, max_iter=25, use_cache=False, label="no-cache")

=== SENZA persist() — equivalente MapReduce ===
  [no-cache] iter  1 | 8.992s | shift: 5.48e-04
  [no-cache] iter  2 | 8.770s | shift: 2.18e-04
  [no-cache] iter  3 | 8.751s | shift: 8.68e-05
  [no-cache] iter  4 | 8.851s | shift: 4.87e-05
  [no-cache] iter  5 | 8.817s | shift: 2.90e-05
  [no-cache] iter  6 | 8.611s | shift: 2.80e-05
  [no-cache] iter  7 | 8.628s | shift: 1.91e-05
  [no-cache] iter  8 | 8.875s | shift: 1.92e-05
  [no-cache] iter  9 | 8.664s | shift: 1.89e-05
  [no-cache] iter 10 | 8.748s | shift: 3.07e-05
  [no-cache] iter 11 | 8.636s | shift: 4.95e-05
  [no-cache] iter 12 | 8.619s | shift: 2.63e-05
  [no-cache] iter 13 | 8.614s | shift: 7.13e-06
  [no-cache] iter 14 | 8.624s | shift: 1.05e-05
  [no-cache] iter 15 | 9.268s | shift: 1.75e-05
  [no-cache] iter 16 | 8.592s | shift: 2.30e-06
  [no-cache] iter 17 | 8.622s | shift: 1.54e-05
  [no-cache] iter 18 | 8.548s | shift: 4.08e-06
  [no-cache] iter 19 | 8.631s | shift: 5.79e-06
  [no-cache] iter 20 | 8.602s | shift: 7

In [ ]:

print("\n=== CON persist() — in-memory computing ===")
cents_ok, times_ok = kmeans(coords_rdd, k=15, max_iter=25, use_cache=True,  label="cached")

print(f"\nTotale senza cache : {sum(times_no):.2f}s")
print(f"Totale con cache   : {sum(times_ok):.2f}s")
print(f"Speedup            : {sum(times_no)/sum(times_ok):.2f}x")


=== CON persist() — in-memory computing ===
  [cached] iter  1 | 8.344s | shift: 5.48e-04
  [cached] iter  2 | 8.873s | shift: 2.18e-04
  [cached] iter  3 | 8.338s | shift: 8.68e-05
  [cached] iter  4 | 8.278s | shift: 4.87e-05
  [cached] iter  5 | 8.260s | shift: 2.90e-05
  [cached] iter  6 | 8.398s | shift: 2.80e-05
  [cached] iter  7 | 8.226s | shift: 1.91e-05
  [cached] iter  8 | 8.467s | shift: 1.92e-05
  [cached] iter  9 | 8.235s | shift: 1.89e-05
  [cached] iter 10 | 8.278s | shift: 3.07e-05
  [cached] iter 11 | 8.249s | shift: 4.95e-05
  [cached] iter 12 | 8.404s | shift: 2.63e-05
  [cached] iter 13 | 8.254s | shift: 7.13e-06
  [cached] iter 14 | 8.328s | shift: 1.05e-05
  [cached] iter 15 | 8.412s | shift: 1.75e-05
  [cached] iter 16 | 8.361s | shift: 2.30e-06
  [cached] iter 17 | 8.288s | shift: 1.54e-05
  [cached] iter 18 | 8.338s | shift: 4.08e-06
  [cached] iter 19 | 8.333s | shift: 5.79e-06
  [cached] iter 20 | 8.264s | shift: 7.98e-06
  [cached] iter 21 | 8.263s | shift

Provo ad usare come benchmark un algoritmo meno cpu-bound:
-  calcolo iterativamente semplicemente un count
- nella versione in memory dalla 2* it in poi uso in memory

In [ ]:
import time

# Versione senza cache: ogni count() rilegge e riesegue tutto il lineage
print("=== Benchmark lineage: SENZA persist() ===")
times_no = []
for i in range(10):
    t0 = time.time()
    _ = coords_rdd.count()
    times_no.append(time.time() - t0)
    print(f"  run {i+1} | {times_no[-1]:.3f}s")

# Versione con cache: prima count() materializza, le successive leggono da RAM
print("\n=== Benchmark lineage: CON persist() ===")
coords_rdd.persist()
times_cache = []
for i in range(10):
    t0 = time.time()
    _ = coords_rdd.count()
    times_cache.append(time.time() - t0)
    print(f"  run {i+1} | {times_cache[-1]:.3f}s")
coords_rdd.unpersist()

print(f"\nMedia senza cache : {sum(times_no[1:])/len(times_no[1:]):.3f}s")
print(f"Media con cache   : {sum(times_cache[1:])/len(times_cache[1:]):.3f}s")
print(f"Speedup           : {sum(times_no[1:])/sum(times_cache[1:]):.2f}x")

=== Benchmark lineage: SENZA persist() ===
  run 1 | 4.095s
  run 2 | 4.210s
  run 3 | 4.076s
  run 4 | 4.047s
  run 5 | 3.995s
  run 6 | 4.020s
  run 7 | 4.045s
  run 8 | 4.024s
  run 9 | 4.075s
  run 10 | 4.063s

=== Benchmark lineage: CON persist() ===
  run 1 | 7.347s
  run 2 | 3.744s
  run 3 | 3.766s
  run 4 | 3.735s
  run 5 | 3.723s
  run 6 | 3.752s
  run 7 | 3.762s
  run 8 | 3.761s
  run 9 | 3.750s
  run 10 | 3.845s

Media senza cache : 4.062s
Media con cache   : 3.760s
Speedup           : 1.08x


### Osservazioni sui risultati 
### Perché lo speedup è presente ma modesto

I risultati mostrano uno speedup di circa **1.05x** per il K-Means e **1.08x**
per il benchmark diretto sul lineage. 

### 1. Ambiente single-node

Spark è progettato per cluster distribuiti. In modalità `local[*]` tutti i
processi girano sulla stessa macchina: non c'è comunicazione di rete, non ci
sono trasferimenti di dati tra nodi, e il parallelismo è limitato ai core
disponibili. In questo contesto il vantaggio di `persist()`, che elimina
il ricalcolo distribuito della lineage, è strutturalmente ridotto.

### 2. OS filesystem cache (page cache)

Su sistemi operativi moderni (Windows incluso), il kernel mantiene in RAM
i blocchi di file letti di recente tramite la **page cache**. Il CSV di Divvy
(~100 MB) viene caricato in RAM dal SO alla prima lettura e vi rimane per
tutta la sessione. Questo significa che le letture successive  anche senza
`persist()`  non toccano mai il disco fisico: il file viene letto dalla RAM
di sistema, in contesti BigData questo non sarebbe possibile.

Di conseguenza, il costo che `persist()` dovrebbe eliminare (l'I/O su disco)
è già basso  grazie alla page cache, rendendo il confronto poco
discriminante.


### Quando il vantaggio di persist() diventa significativo

Il beneficio reale dell'in-memory computing emerge in tre scenari:

- **Cluster distribuito**: i dati risiedono su nodi diversi; senza cache ogni
  iterazione richiede shuffle di rete, che è ordini di grandezza più lento
  della RAM locale.
- **Dataset che non entra nella page cache del SO**: file da decine di GB
  vengono riletti dal disco ad ogni iterazione senza `persist()`.


Il benchmark del lineage (`count()` ripetuto) mostra chiaramente il
comportamento atteso nella cella precedente: la prima esecuzione con
`persist()` è più lenta (7.3s, materializzazione), ma tutte le successive
scendono a ~3.7s rispetto a ~4.0s senza cache — una differenza piccola ma
consistente, interamente dovuta alla page cache del SO che riduce il gap.


---

## Parte 2 — Spark Streaming

Spark offre una API per dati che arrivano in modalità streaming:
- **Structured Streaming**: API moderna che astrae il flusso come una tabella
  che cresce nel tempo. Usa DataFrame internamente il modello funaziona a micro-batch discreti, ognuno processato come
  un'operazione su dati strutturati. 

### Architettura

Il flusso continuo di dati viene suddiviso in micro-batch discreti, ognuno
processato come una query su un DataFrame in arrivo:

```
flusso continuo  →  [DF_t0] [DF_t1] [DF_t2] ...
                       ↑       ↑       ↑
                    batch_0  batch_1  batch_2
                    (5s)     (5s)     (5s)
```

### Simulazione del flusso

Simuliamo il flusso con la strategia **directory watching**: un thread in
background copia porzioni del dataset Divvy in una cartella ogni 5 secondi.
Spark Streaming monitora la cartella e processa ogni nuovo file come un
micro-batch.

### Perché non un DBMS relazionale

Un RDBMS classico non è progettato per dati in movimento: per sapere cosa
è cambiato bisogna interrogarlo esplicitamente (pull), simulando lo streaming
con polling continuo. Questo introduce latenza, spreca risorse su query
ripetute che spesso non trovano nuovi dati, e non scala orizzontalmente
quando il volume aumenta. Le aggregazioni su finestre temporali scorrevoli
richiederebbero query complesse con self-join o indici costosi da mantenere
aggiornati ad ogni insert. Spark Structured Streaming gestisce tutto questo
nativamente.

In [5]:
from pyspark.streaming import StreamingContext
import threading, shutil, time, os

In [6]:
''' 
    Setup per simulare un flusso di dati:
'''

STREAM_DIR   = "./stream_input"
CHECKPOINT   = "./stream_checkpoint"
CHUNK_SIZE   = 2000   # righe per batch
BATCH_INTERVAL = 5    # secondi

# Pulisci da run precedenti
for d in [STREAM_DIR, CHECKPOINT]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)

# Carica tutte le righe del CSV (senza header)
with open(CSV_PATH, "r", encoding="utf-8") as f:
    all_lines = f.readlines()
header_line = all_lines[0]
data_lines  = all_lines[1:]

print(f"Righe totali nel dataset: {len(data_lines):,}")
print(f"Chunk size: {CHUNK_SIZE} righe → ~{len(data_lines)//CHUNK_SIZE} batch disponibili")

Righe totali nel dataset: 821,276
Chunk size: 2000 righe → ~410 batch disponibili


Definizione del corpo del thread producer di chunks

In [ ]:
import socket, threading, time, os

SOCKET_HOST    = "localhost"
SOCKET_PORT    = 9999
BATCH_INTERVAL = 5
CHUNK_LINES    = 500

producer_active = threading.Event()
producer_active.set()

def socket_producer(lines, chunk_size, port, interval, n_chunks=15):
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((SOCKET_HOST, port))
    server.listen(1)
    server.settimeout(30)
    print(f"  [producer] in ascolto su porta {port}...")
    
    try:
        conn, addr = server.accept()
        print(f"  [producer] Spark connesso da {addr}")
        
        for i in range(n_chunks):
            if not producer_active.is_set():
                break
            chunk = lines[i * chunk_size : (i+1) * chunk_size]
            for line in chunk:
                try:
                    conn.sendall((line.strip() + "\n").encode("utf-8"))
                except BrokenPipeError:
                    print("  [producer] connessione chiusa.")
                    return
            print(f"  [producer] chunk {i+1:02d} inviato → {len(chunk)} righe")
            time.sleep(interval)
    except socket.timeout:
        print("  [producer] timeout attesa connessione.")
    finally:
        server.close()
    print("  [producer] terminato.")

threading.Thread(
    target=socket_producer,
    args=(data_lines, CHUNK_LINES, SOCKET_PORT, BATCH_INTERVAL, 15),
    daemon=True
).start()

print("Producer avviato. Aspetta la connessione di Spark...")

In [ ]:
# Avvia il produttore in background
threading.Thread(
    target=socket_producer,
    args=(data_lines, CHUNK_LINES, SOCKET_PORT, BATCH_INTERVAL, 15),
    daemon=True
).start()

print("Producer avviato. Aspetta la connessione di Spark...")

  [producer] chunk 01 scritto → 2000 righe


In [ ]:
# Se si vuole farlo terminare prima del completamento:
producer_active.clear()

  [producer] completato.
  [producer] scritto chunk 6
  [producer] scritto chunk 7
  [producer] scritto chunk 8
  [producer] scritto chunk 9


In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import time

spark = SparkSession(sc)

# Legge righe raw dal socket
raw_df = (
    spark.readStream
    .format("socket")
    .option("host", SOCKET_HOST)
    .option("port", SOCKET_PORT)
    .load()
)

# Parsing: split della riga CSV
def parse_row(df):
    return df.select(
        split(col("value"), ",").alias("fields")
    ).select(
        col("fields")[4].alias("start_station_name"),
        col("fields")[8].cast(DoubleType()).alias("start_lat"),
        col("fields")[9].cast(DoubleType()).alias("start_lng"),
    ).filter(
        (col("start_lat").between(41.6, 42.1)) &
        (col("start_lng").between(-87.9, -87.5)) &
        (col("start_station_name").isNotNull()) &
        (col("start_station_name") != "")
    )

parsed_df = parse_row(raw_df)

trip_counts = (
    parsed_df
    .groupBy("start_station_name")
    .count()
    .orderBy(col("count").desc())
)

def process_batch(batch_df, batch_id):
    try:
        if batch_df.rdd.isEmpty():
            print(f"  [batch {batch_id}] nessun dato ancora...")
            return
        total = batch_df.agg({"count": "sum"}).collect()[0][0]
        top   = batch_df.limit(5).collect()
        print(f"\n  [batch {batch_id} | {time.strftime('%H:%M:%S')}] {total} trip nel batch")
        for row in top:
            bar = "█" * min(row['count'] // 5, 40)
            name = (row['start_station_name'] or "?").strip('"')[:40]
            print(f"    {name:<40} {row['count']:4d}  {bar}")
    except Exception as e:
        print(f"  [ERRORE batch {batch_id}]: {e}")

query = (
    trip_counts.writeStream
    .outputMode("complete")
    .foreachBatch(process_batch)
    .trigger(processingTime=f"{BATCH_INTERVAL} seconds")
    .start()
)

print("Pipeline definita.")

Pipeline definita.


In [13]:
print("Streaming avviato per ~90 secondi...")
print("="*60)
query.awaitTermination(90)
query.stop()
print("\nStreaming completato.")

Streaming avviato per ~90 secondi...


StreamingQueryException: [STREAM_FAILED] Query [id = 837680d5-befb-4c24-bab4-f41e2e581ebe, runId = be025435-a7b7-4fb5-9bb9-3971f925ce52] terminated with exception: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)' SQLSTATE: XXKST
=== Streaming Query ===
Identifier: [id = 837680d5-befb-4c24-bab4-f41e2e581ebe, runId = be025435-a7b7-4fb5-9bb9-3971f925ce52]
Current Committed Offsets: {}
Current Available Offsets: {}

Current State: ACTIVE
Thread State: RUNNABLE

Logical Plan:
~WriteToMicroBatchDataSourceV1 ForeachBatchSink, 837680d5-befb-4c24-bab4-f41e2e581ebe, Complete
+- ~Sort [count#5L DESC NULLS LAST], true
   +- ~Aggregate [start_station_name#2], [start_station_name#2, count(1) AS count#5L]
      +- ~Filter (((((start_lat#3 >= 41.6) AND (start_lat#3 <= 42.1)) AND ((start_lng#4 >= -87.9) AND (start_lng#4 <= -87.5))) AND isnotnull(start_station_name#2)) AND NOT (start_station_name#2 = ))
         +- ~Project [fields#1[4] AS start_station_name#2, cas

In [ ]:
producer_active.clear()
try:
    query.stop()
except:
    pass
print("Cleanup completato.")

Directory temporanee rimosse.
